# Natural associations and controlled processes
## Natural data
Only unmodified electricity data belong in this section. Trailing mean/variance versus forecast errors describe associations, not known drift or causation. No electricity perturbations are applied.

In [ ]:
from pathlib import Path
import os, sys, json, time
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'research').is_dir())
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
os.environ.setdefault('OMP_NUM_THREADS','1')
os.environ.setdefault('OPENBLAS_NUM_THREADS','1')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from research.experiment import config, hardware, run_stream, metrics
from research.data import acquire, parse_tsf, audit, select
from research.synthetic import processes, change_analysis
from research.inspired_experiment import run_inspired, inspired_config
from research.inspired_study import compare_streams, controlled
from research.validation import source, electricity_source
MODE = os.environ.get('EXPERIMENT_MODE','quick')
cfg = config(MODE)
OUT = Path(os.environ.get('DNH_RESULTS_DIR',ROOT/'results'/MODE))
OUT.mkdir(parents=True, exist_ok=True)
print(hardware()); print(cfg)
def stream_provenance(sid, values, generator_seed=17):
    if series:
        return electricity_source({'id':sid,'values':values},verified)
    return source('synthetic',generator_seed=generator_seed,process='stationary')


In [ ]:
paths=list((OUT/'comparison').glob('*/predictions.csv'))
natural=[]
raw_series={}
if list((ROOT/'data/raw').glob('*.zip')):
    _, verified_series=parse_tsf(acquire(ROOT/'data/raw',manual=True))
    raw_series={s['id']:s['values'] for s in verified_series}
for path in paths:
    if not path.parent.name.startswith('synthetic_'):
        p=pd.read_csv(path); g=p[p.model=='ARIMA'].copy()
        history=pd.Series(raw_series[path.parent.name])
        g['trailing_target_mean']=history.shift(1).rolling(24).mean().iloc[g.t.to_numpy()].to_numpy()
        g['trailing_target_variance']=history.shift(1).rolling(24).var().iloc[g.t.to_numpy()].to_numpy()
        g['series']=path.parent.name; natural.append(g)
if natural:
    association=pd.concat(natural); association.to_csv(OUT/'natural_associations.csv',index=False)
    display(association.groupby('series')[['abs_error','trailing_target_mean','trailing_target_variance']].corr())
else: print('Natural-data analysis NOT RUN: verified electricity unavailable.')

## Controlled data (separate from electricity)
Stationary control is zero-mean Gaussian AR(1), coefficient 0.6, innovation SD 1, initialized from its stationary distribution. Paired runs share initial value and innovations. Abrupt: coefficient switches to −0.6. Gradual: ramps from 0.6 to −0.6. Recurring: alternates regimes. Noise: coefficient stays 0.6, innovation SD changes to 2. Known change points lie one third into evaluation.

Recovery is the first completed 12-point post-change error window no worse than 1.1× the matched control, for three consecutive windows. For relationship changes, no crossing means non-recovery within the observed horizon; this relative criterion can occasionally indicate apparent recovery by chance, especially in tiny quick runs. Increased noise is now reported separately as noise sensitivity: recovery is not applicable. Known innovation variance, oracle RMSE, noise-normalized MAE and realized excess MSE separate noise-floor changes from predictor error. Report excess errors separately. Quick horizons are too short for robust adaptation inference.

In [ ]:
all_analysis=[]
for seed in cfg['seeds']:
    synthetic=processes(cfg,seed); runs={}
    for kind,s in synthetic.items():
        pred,report=run_stream(s['values'],cfg,OUT/'controlled'/str(seed)/kind,provenance=source('synthetic',generator_seed=seed,process=kind))
        runs[kind]=pred
    analysis=change_analysis(runs,synthetic['abrupt']['change_point'],oracle=synthetic)
    analysis['process_seed']=seed; all_analysis.append(analysis)
    for kind,p in runs.items():
        g=p[p.model=='ARIMA']; plt.plot(g.t,g.abs_error.rolling(12).mean(),label=kind)
    plt.axvline(synthetic['abrupt']['change_point'],color='k',ls='--')
    plt.legend(); plt.ylabel('ARIMA trailing MAE'); plt.title(f'Controlled seed {seed}')
    plt.savefig(OUT/f'controlled_{seed}.png',bbox_inches='tight'); plt.show()
analysis=pd.concat(all_analysis,ignore_index=True)
analysis.to_csv(OUT/'controlled_analysis.csv',index=False); display(analysis)
print('These are actual baseline diagnostics, not DNH results or completed electricity experiments.')

## DNH-inspired controlled comparison (separate from electricity)
Use matched innovation streams and initialization for each process control. Report process/generator seed separately from memory-model initialization seed. Adaptive and fixed counterparts see the same target histories. The existing predeclared recovery criterion applies unchanged; non-recovery is explicit. Three model seeds on one quick generator seed do not establish variation across data-generating processes. No electricity perturbations are used.

In [ ]:
inspired_control=controlled(cfg,OUT/'inspired_controlled')
display(inspired_control)
print('Non-recovery counts by model:',inspired_control[inspired_control.recovery_applicable].groupby('model').recovered_within_horizon.apply(lambda v:int((~v.astype(bool)).sum())).to_dict())
# Natural-data associations, descriptively per customer/seed; no drift labels or causality.
associations=[]
for sid in raw_series:
    paths=list((OUT/'inspired_comparison'/sid).glob('*/*/predictions.csv'))
    history=pd.Series(raw_series[sid])
    for path in paths:
        p=pd.read_csv(path);p['series']=sid
        p['trailing_target_mean']=history.shift(1).rolling(24).mean().iloc[p.t.to_numpy()].to_numpy()
        p['trailing_target_variance']=history.shift(1).rolling(24).var().iloc[p.t.to_numpy()].to_numpy()
        associations.append(p)
if associations:
    a=pd.concat(associations,ignore_index=True);a.to_csv(OUT/'inspired_natural_associations.csv',index=False)
    display(a.groupby(['series','model','seed'])[['abs_error','trailing_target_mean','trailing_target_variance']].corr())
else: print('Run notebook 03 first for inspired natural associations.')